# <font color="#418FDE" size="6.5" uppercase>**Semi Supervised**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Represent labeled and unlabeled samples for semi-supervised estimators. 
- Apply self-training with threshold and k-best pseudo-labeling strategies. 
- Use graph-based label propagation and spreading methods and evaluate their behavior. 


## **1. Semi Supervised Setup**

### **1.1. Unlabeled Sample Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_01_01.jpg?v=1784014907" width="250">



>* Mix labeled and unlabeled samples together
>* Mark unknown targets with a special placeholder

>* Choose a marker outside valid classes
>* Use it consistently for all unlabeled samples

>* Process all features the same way
>* Mark unlabeled targets to reveal data structure



In [ ]:
#@title Python Code - Unlabeled Sample Encoding

# This example encodes unlabeled samples for semi-supervised learning.
# The target array uses negative one for unknown labels.
# The output shows labeled and unlabeled counts clearly.

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
import sklearn

# Load a small packaged classification dataset.
iris = load_iris()
features = iris.data[:12, :2]
true_labels = iris.target[:12]

# Copy labels before replacing some with the unlabeled marker.
semi_supervised_labels = true_labels.copy()
unlabeled_marker = -1
semi_supervised_labels[[2, 5, 8, 11]] = unlabeled_marker

# Validate that the marker is not a real class label.
real_classes = np.unique(true_labels)
if unlabeled_marker in real_classes:
    raise ValueError("Choose an unlabeled marker outside the real classes.")

# Build a compact table showing the encoding idea.
status = np.where(
    semi_supervised_labels == unlabeled_marker,
    "unlabeled",
    "labeled",
)

example_table = pd.DataFrame(
    {
        "sepal_length": features[:, 0],
        "sepal_width": features[:, 1],
        "target": semi_supervised_labels,
        "status": status,
    }
)

# Count trusted labels separately from unknown targets.
labeled_count = int(np.sum(semi_supervised_labels != unlabeled_marker))
unlabeled_count = int(np.sum(semi_supervised_labels == unlabeled_marker))

print("scikit-learn version:", sklearn.__version__)
print("Unlabeled marker:", unlabeled_marker)
print("Real class labels:", real_classes.tolist())
print("Labeled samples:", labeled_count)
print("Unlabeled samples:", unlabeled_count)
print(example_table.head(5).to_string(index=False))



### **1.2. Self Training Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_01_02.jpg?v=1784014906" width="250">



>* Train first on trusted labeled samples
>* Add confident pseudo-labels from unlabeled data

>* Learn from labeled messages first
>* Pseudo-label confident unlabeled examples later

>* Control pseudo-labels to avoid reinforcing mistakes
>* Track labeled, unlabeled, and pseudo-labeled roles



In [ ]:
#@title Python Code - Self Training Basics

# This example marks unlabeled samples for self-training.
# Unlabeled targets use negative one in scikit-learn.
# The output compares trusted and pseudo labels.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y_true = iris.target

# Keep only a few trusted labels per class.
y_semi = np.full(y_true.shape, -1, dtype=int)
for class_id in np.unique(y_true):
    class_positions = np.flatnonzero(y_true == class_id)
    y_semi[class_positions[:5]] = class_id

# Validate the semi-supervised target representation.
labeled_count = int(np.sum(y_semi != -1))
unlabeled_count = int(np.sum(y_semi == -1))
if labeled_count == 0 or unlabeled_count == 0:
    raise ValueError("Need both labeled and unlabeled samples.")

# Wrap a supervised model inside self-training.
base_model = LogisticRegression(max_iter=200, random_state=42)
self_trainer = SelfTrainingClassifier(
    base_model,
    threshold=0.80,
    criterion="threshold",
)

# Fit using trusted labels and negative-one unlabeled markers.
self_trainer.fit(X, y_semi)
pseudo_count = int(np.sum((y_semi == -1) & (self_trainer.transduction_ != -1)))
accuracy = self_trainer.score(X, y_true)

print("scikit-learn version:", sklearn.__version__)
print("Trusted human labels:", labeled_count)
print("Initially unlabeled samples:", unlabeled_count)
print("Pseudo-labeled by self-training:", pseudo_count)
print("Accuracy against hidden true labels:", round(accuracy, 3))



### **1.3. Choosing Base Estimators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_01_03.jpg?v=1784014909" width="250">



>* Base estimator links labels to unlabeled patterns
>* Learn confidently without spreading unreliable labels

>* Choose estimators with reliable confidence scores
>* Avoid reinforcing wrong pseudo-labels

>* Match estimator type to data and assumptions
>* Validate carefully before pseudo-labeling unlabeled samples



In [ ]:
#@title Python Code - Choosing Base Estimators

# This example compares base estimators for self-training.
# Unlabeled samples use minus one target labels.
# Confidence quality affects which pseudo-labels are accepted.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.tree import DecisionTreeClassifier

# Create a small classification dataset with known true labels.
features, true_labels = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42,
)

# Keep only a few labels and mark the rest unlabeled.
semi_labels = np.full(true_labels.shape, -1)
labeled_indices = np.r_[0:10, 150:160]
semi_labels[labeled_indices] = true_labels[labeled_indices]

# Validate the semi-supervised label representation.
if np.sum(semi_labels != -1) != 20:
    raise ValueError("Expected exactly 20 labeled samples.")

# Choose two base estimators that can provide probabilities.
base_estimators = {
    "Logistic regression": LogisticRegression(random_state=42, max_iter=200),
    "Decision tree": DecisionTreeClassifier(max_depth=3, random_state=42),
}

print("scikit-learn version:", sklearn.__version__)
print("Labeled samples:", int(np.sum(semi_labels != -1)))
print("Unlabeled samples:", int(np.sum(semi_labels == -1)))

# Fit self-training once for each base estimator.
for name, estimator in base_estimators.items():
    model = SelfTrainingClassifier(
        estimator=estimator,
        threshold=0.85,
        max_iter=10,
    )
    model.fit(features, semi_labels)
    accepted = int(np.sum(model.transduction_ != -1))
    accuracy = model.score(features, true_labels)
    print(name + " accepted:", accepted, "accuracy:", round(accuracy, 3))



## **2. Pseudo Labeling Strategies**

### **2.1. Confidence Thresholds**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_02_01.jpg?v=1784014899" width="250">



>* Accept only high-confidence pseudo labels
>* Delay uncertain or noisy examples

>* High thresholds give cleaner but fewer labels
>* Low thresholds add data but risk errors

>* Confidence can mislead on changing data
>* Validate thresholds and monitor class balance



In [ ]:
#@title Python Code - Confidence Thresholds

# Demonstrate confidence thresholds in self-training.
# Compare strict and relaxed pseudo-label acceptance.
# Show how thresholds change accepted examples.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import SelfTrainingClassifier

# Create a small classification dataset with known true labels.
features, labels = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    n_classes=3,
    random_state=42,
)

# Keep a test set fully labeled for fair evaluation.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.35,
    stratify=labels,
    random_state=42,
)

# Hide most training labels by marking them as unlabeled.
rng = np.random.default_rng(42)
semi_labels = np.full(train_labels.shape, -1)

# Keep only a few true labels per class.
for class_value in np.unique(train_labels):
    class_positions = np.where(train_labels == class_value)[0]
    chosen_positions = rng.choice(class_positions, size=12, replace=False)
    semi_labels[chosen_positions] = train_labels[chosen_positions]

# Validate that labeled and unlabeled samples are both present.
labeled_count = int(np.sum(semi_labels != -1))
unlabeled_count = int(np.sum(semi_labels == -1))

if labeled_count == 0 or unlabeled_count == 0:
    raise ValueError("Need both labeled and unlabeled training samples.")

# Train the same base model with two confidence thresholds.
thresholds = [0.75, 0.95]
results = []

for threshold in thresholds:
    base_model = LogisticRegression(max_iter=500, random_state=42)
    self_trainer = SelfTrainingClassifier(
        base_model,
        threshold=threshold,
        criterion="threshold",
    )
    self_trainer.fit(train_features, semi_labels)

    pseudo_count = int(np.sum(self_trainer.transduction_ != -1) - labeled_count)
    predictions = self_trainer.predict(test_features)
    accuracy = accuracy_score(test_labels, predictions)
    results.append((threshold, pseudo_count, accuracy))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Initially labeled samples: {labeled_count}")
print(f"Initially unlabeled samples: {unlabeled_count}")

for threshold, pseudo_count, accuracy in results:
    print(
        f"Threshold {threshold}: accepted {pseudo_count} pseudo-labels, "
        f"test accuracy {accuracy:.3f}"
    )



### **2.2. K Best Pseudo Labels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_02_02.jpg?v=1784014901" width="250">



>* Select only the most confident pseudo labels
>* Grow training data cautiously over time

>* Choose k carefully to balance learning and noise
>* Increase pseudo labels gradually as confidence improves

>* Control labeling pace, but monitor selection bias
>* Use balancing and validation for broader improvement



In [ ]:
#@title Python Code - K Best Pseudo Labels

# Demonstrate k best pseudo labeling.
# Add only the most confident predictions.
# Compare cautious and aggressive self training.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small classification dataset with known labels.
features, labels = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    n_classes=2,
    random_state=42,
)

# Split once so the test set stays truly unseen.
train_features, test_features, train_labels, test_labels = train_test_split(
    features,
    labels,
    test_size=0.35,
    stratify=labels,
    random_state=42,
)

# Scale features using only the training portion.
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)

# Keep only a few labels and hide the rest.
rng = np.random.default_rng(42)
labeled_indices = []
for class_value in np.unique(train_labels):
    class_indices = np.where(train_labels == class_value)[0]
    chosen = rng.choice(class_indices, size=8, replace=False)
    labeled_indices.extend(chosen.tolist())

# Store unlabeled indices separately for pseudo labeling.
labeled_indices = np.array(sorted(labeled_indices))
all_train_indices = np.arange(len(train_labels))
unlabeled_indices = np.setdiff1d(all_train_indices, labeled_indices)

if len(labeled_indices) != 16 or len(unlabeled_indices) == 0:
    raise ValueError("Expected a small labeled set and unlabeled pool.")

# This helper adds the k most confident pseudo labels.
def run_k_best_self_training(k_per_round, rounds):
    current_labeled = labeled_indices.copy()
    current_unlabeled = unlabeled_indices.copy()
    pseudo_labels = train_labels.copy()

    for round_number in range(rounds):
        model = LogisticRegression(max_iter=300, random_state=42)
        model.fit(train_features[current_labeled], pseudo_labels[current_labeled])

        probabilities = model.predict_proba(train_features[current_unlabeled])
        confidence = probabilities.max(axis=1)
        predicted_labels = model.classes_[probabilities.argmax(axis=1)]

        add_count = min(k_per_round, len(current_unlabeled))
        best_positions = np.argsort(confidence)[-add_count:]
        selected_indices = current_unlabeled[best_positions]

        pseudo_labels[selected_indices] = predicted_labels[best_positions]
        current_labeled = np.concatenate([current_labeled, selected_indices])
        current_unlabeled = np.delete(current_unlabeled, best_positions)

    final_model = LogisticRegression(max_iter=300, random_state=42)
    final_model.fit(train_features[current_labeled], pseudo_labels[current_labeled])
    test_predictions = final_model.predict(test_features)

    accuracy = accuracy_score(test_labels, test_predictions)
    added_count = len(current_labeled) - len(labeled_indices)
    return accuracy, added_count

# Train a baseline using only the original labeled examples.
baseline_model = LogisticRegression(max_iter=300, random_state=42)
baseline_model.fit(train_features[labeled_indices], train_labels[labeled_indices])
baseline_accuracy = accuracy_score(test_labels, baseline_model.predict(test_features))

# Compare two k values to show the pace of pseudo labeling.
small_k_accuracy, small_k_added = run_k_best_self_training(k_per_round=5, rounds=6)
large_k_accuracy, large_k_added = run_k_best_self_training(k_per_round=30, rounds=6)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original labeled samples: {len(labeled_indices)}")
print(f"Baseline accuracy: {baseline_accuracy:.3f}")
print(f"k=5 added {small_k_added} pseudo labels, accuracy: {small_k_accuracy:.3f}")
print(f"k=30 added {large_k_added} pseudo labels, accuracy: {large_k_accuracy:.3f}")



### **2.3. Avoiding Confirmation Bias**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_02_03.jpg?v=1784014904" width="250">



>* Early pseudo labels can reinforce model biases
>* Treat pseudo labels as imperfect training signals

>* Use thresholds and k-best cautiously
>* Monitor class balance to avoid bias

>* Add pseudo labels gradually and validate often
>* Keep labels distinct and select diverse examples



## **3. Graph Based Labeling**

### **3.1. Label Propagation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_03_01.jpg?v=1784014911" width="250">



>* Similar samples form connected graph nodes
>* Known labels spread to unlabeled neighbors

>* Strong graph links spread labels effectively
>* Noisy similarities can spread wrong labels

>* Labeled nodes stay fixed during updates
>* Validate graph quality and error patterns



In [ ]:
#@title Python Code - Label Propagation

# Demonstrate label propagation with few labeled samples.
# Nearby points share labels through a graph.
# The plot shows inferred labels and confidence.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.semi_supervised import LabelPropagation
from sklearn.metrics import accuracy_score

# Create two clear clusters for semi-supervised classification.
features, true_labels = make_blobs(
    n_samples=160, centers=2, cluster_std=1.25, random_state=42
)

# Start with every sample marked as unlabeled.
training_labels = np.full(true_labels.shape, -1)
training_labels[0] = true_labels[0]
training_labels[1] = true_labels[1]

# Ensure the two seed labels represent different classes.
if training_labels[0] == training_labels[1]:
    second_class_index = np.where(true_labels != training_labels[0])[0][0]
    training_labels[1] = -1
    training_labels[second_class_index] = true_labels[second_class_index]

# Fit graph-based label propagation using all feature points.
model = LabelPropagation(kernel="knn", n_neighbors=10, max_iter=1000)
model.fit(features, training_labels)

# Compare propagated labels with the hidden true labels.
predicted_labels = model.transduction_
accuracy = accuracy_score(true_labels, predicted_labels)

# Confidence is the largest class probability per sample.
confidence = model.label_distributions_.max(axis=1)
mean_unlabeled_confidence = confidence[training_labels == -1].mean()

print("scikit-learn version:", sklearn.__version__)
print("Initially labeled samples:", int(np.sum(training_labels != -1)))
print("Accuracy after propagation:", round(accuracy, 3))
print("Mean confidence on originally unlabeled samples:", round(mean_unlabeled_confidence, 3))

# Mark original labeled seeds with larger black-edged points.
seed_mask = training_labels != -1
plt.figure(figsize=(7, 5))
plt.scatter(
    features[:, 0], features[:, 1], c=predicted_labels, cmap="coolwarm", alpha=0.65
)
plt.scatter(
    features[seed_mask, 0], features[seed_mask, 1], c=training_labels[seed_mask],
    cmap="coolwarm", edgecolors="black", s=160, marker="*", label="labeled seeds"
)
plt.title("Label Propagation from Two Labeled Seeds")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()



### **3.2. Label Spreading**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_03_02.jpg?v=1784014913" width="250">



>* Spreads known labels through similarity graphs
>* Uses unlabeled data when labels are scarce

>* Softens labels instead of fixing them
>* Reduces noise through graph consistency

>* Works best with meaningful similarity clusters
>* Evaluate accuracy, confidence, balance, and boundaries



In [ ]:
#@title Python Code - Label Spreading

# Demonstrate label spreading with few labeled samples.
# Unlabeled points receive labels through graph smoothing.
# The plot shows predictions across two clusters.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons

from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score

# Create a small curved dataset with two natural classes.
features, true_labels = make_moons(
    n_samples=300, noise=0.12, random_state=42
)

# Keep a test set with true labels for evaluation.
train_features, test_features, train_true, test_true = train_test_split(
    features, true_labels, test_size=0.35, stratify=true_labels, random_state=42
)

# Hide most training labels using -1 for unlabeled samples.
semi_labels = np.full(train_true.shape, -1, dtype=int)

# Reveal only a few labels from each class.
for class_value in [0, 1]:
    class_positions = np.where(train_true == class_value)[0]
    semi_labels[class_positions[:6]] = class_value

# Validate that both classes have labeled anchor points.
labeled_count = int(np.sum(semi_labels != -1))
if labeled_count != 12:
    raise ValueError("Expected exactly twelve labeled training samples.")

# Fit label spreading on labeled and unlabeled training samples.
model = LabelSpreading(kernel="rbf", gamma=20, alpha=0.2, max_iter=100)
model.fit(train_features, semi_labels)

# Evaluate predictions on held-out labels never used for fitting.
test_predictions = model.predict(test_features)
test_accuracy = accuracy_score(test_true, test_predictions)

# Compare inferred training labels with the hidden true labels.
train_predictions = model.transduction_
train_accuracy = accuracy_score(train_true, train_predictions)

print("scikit-learn version:", sklearn.__version__)
print("Labeled training samples:", labeled_count, "of", len(train_true))
print("Training label recovery accuracy:", round(train_accuracy, 3))
print("Held-out test accuracy:", round(test_accuracy, 3))

# Plot the learned smooth regions and the few labeled anchors.
fig, ax = plt.subplots(figsize=(7, 5))

scatter = ax.scatter(
    train_features[:, 0], train_features[:, 1], c=train_predictions, cmap="coolwarm", s=35
)

# Mark the original labeled examples with black outlines.
anchor_mask = semi_labels != -1
ax.scatter(
    train_features[anchor_mask, 0], train_features[anchor_mask, 1], facecolors="none", edgecolors="black", s=120, label="given labels"
)

ax.set_title("Label Spreading from 12 Labeled Samples")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(loc="best")
plt.show()



### **3.3. Graph Kernel Choices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_19/Lecture_B/image_03_03.jpg?v=1784014915" width="250">



>* Graph kernels define meaningful sample connections
>* Poor connections spread labels incorrectly

>* Nearest neighbors preserve local structure
>* RBF scale controls smoothing and fragmentation

>* Check graphs match real problem structure
>* Kernels shape similarity and label flow



In [ ]:
#@title Python Code - Graph Kernel Choices

# Compare graph kernels for semi-supervised labeling.
# Kernel choice changes how labels spread.
# The plot shows different decision regions.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler

# Create two curved classes with many unlabeled samples.
features, true_labels = make_moons(
    n_samples=240,
    noise=0.12,
    random_state=42,
)

# Scale features so distance-based graph kernels are fair.
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Keep only a few labels and mark the rest as unknown.
labeled_labels = np.full(true_labels.shape, -1)
for class_value in [0, 1]:
    class_positions = np.where(true_labels == class_value)[0]
    labeled_labels[class_positions[:4]] = class_value

# Validate that both classes have labeled examples.
known_count = np.sum(labeled_labels != -1)
if known_count != 8:
    raise ValueError("Expected exactly eight labeled examples.")

# Fit two graph kernels on the same semi-supervised data.
knn_model = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2)
knn_model.fit(features, labeled_labels)

rbf_model = LabelSpreading(kernel="rbf", gamma=8, alpha=0.2)
rbf_model.fit(features, labeled_labels)

# Compare how well each kernel recovered hidden labels.
knn_accuracy = accuracy_score(true_labels, knn_model.transduction_)
rbf_accuracy = accuracy_score(true_labels, rbf_model.transduction_)

print(f"scikit-learn version: {sklearn_version}")
print(f"Labeled samples used: {known_count} of {len(true_labels)}")
print(f"k-nearest-neighbor graph accuracy: {knn_accuracy:.3f}")
print(f"RBF graph accuracy: {rbf_accuracy:.3f}")

# Plot the data colored by the RBF graph predictions.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=rbf_model.transduction_,
    cmap="coolwarm",
    s=35,
    alpha=0.75,
)

# Highlight the few points that originally had labels.
labeled_mask = labeled_labels != -1
ax.scatter(
    features[labeled_mask, 0],
    features[labeled_mask, 1],
    c=true_labels[labeled_mask],
    cmap="coolwarm",
    s=130,
    edgecolor="black",
    linewidth=1.5,
    label="original labels",
)

ax.set_title("Label Spreading with an RBF Graph Kernel")
ax.set_xlabel("scaled feature 1")
ax.set_ylabel("scaled feature 2")
ax.legend(loc="best")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Semi Supervised**</font>


In this lecture, you learned to:
- Represent labeled and unlabeled samples for semi-supervised estimators. 
- Apply self-training with threshold and k-best pseudo-labeling strategies. 
- Use graph-based label propagation and spreading methods and evaluate their behavior. 

In the next Module (Module 20), we will go over 'Feature Selection'